# Ручная проверка ответов по TLS
Выполните **Run → Run All Cells**. Сначала откроется сохранённый пример. Чтобы задать свой вопрос, перейдите к разделу **Свой вопрос**, поменяйте параметры и нажмите **Shift + Enter**. Ноутбук запускает тот же `rag.py ask`, что и терминал.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import textwrap

from corpus import ROOT, read_corpus

sources = read_corpus()

def show_answer(row):
    print(row["question"])
    print(f'Метод {row["method"]} · {row["status"]} · генерация при первом прогоне {row["duration_s"]} с')
    if row["status"] == "insufficient":
        print("Модель сообщила, что в найденных фрагментах недостаточно данных.")
    elif row["status"] == "invalid_evidence":
        print("Ответ отклонён при проверке формата или ссылок.")
        print(row["raw_output"])

    for claim in row["claims"]:
        start, end = claim["start_byte"], claim["end_byte"]
        quote = sources[claim["rfc"]][start:end].decode()
        if quote != claim["quote"]:
            raise ValueError("Цитата в результате отличается от оригинала RFC")
        print()
        print(claim["text"])
        print(f'RFC {claim["rfc"]} §{claim["section"]}, байты [{start}, {end})')
        print(textwrap.fill(" ".join(quote.split()), width=90, initial_indent="  ", subsequent_indent="  "))

def ask(question, method):
    python = ROOT / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    if not python.is_file():
        raise RuntimeError("Подготовьте окружение проекта по разделу «Повторный эксперимент» в README.")
    env = os.environ.copy()
    if (ROOT / ".runtime" / "ollama-models").exists():
        env.setdefault("OLLAMA_HOST", "127.0.0.1:11439")
    result = subprocess.run([str(python), "rag.py", "ask", question, "--method", method],
                            cwd=ROOT, env=env, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "Не удалось выполнить rag.py ask")
    return json.loads(result.stdout)

## Сохранённый пример
Первый пример показывает, что происходит, когда нужное правило продолжается на следующей странице RFC. Чтобы увидеть отказ на вопрос о Safari, замените `live-rc4` на `live-safari`. Примеры читаются с диска.

In [ ]:
example_id = "live-rc4"
examples = [json.loads(line) for line in (ROOT / "results/examples-D.jsonl").read_text().splitlines()]
example = next(row for row in examples if row["id"] == example_id)
show_answer(example)

## Свой вопрос
Задайте `run_live = True`, укажите текст и методы, затем выполните ячейку. `A` использует векторный поиск, `B` добавляет BM25, `C` переводит вопрос для BM25, `D` дополняет B продолжением раздела на следующей странице. Методы запускаются по очереди. Для живого запроса нужен локальный Ollama с моделью проекта. Повторный вызов может использовать кеш.

In [ ]:
run_live = False
question = "По RFC 7465 клиент предлагает только наборы RC4. Что обязан сделать сервер?"
methods = ["B", "D"]

answers = []
if run_live:
    for method in methods:
        row = ask(question, method)
        answers.append(row)
        show_answer(row)
        print()
else:
    print("Для своего вопроса включите run_live и выполните эту ячейку ещё раз.")

## Что увидела модель
Выполните ячейку после запроса. Она показывает пять фрагментов в порядке выдачи. Проверьте, нашёлся ли абзац с нужным условием, затем прочитайте цитату у каждого утверждения. Для примера про RC4 проверьте отдельно значение слов *only*, *MUST* и *MAY*.

In [ ]:
for row in answers or [example]:
    print(f'Метод {row["method"]} · {row["question"]}')
    for rank, key in enumerate(row["retrieved"], 1):
        document, interval = key.split(":")
        start, end = map(int, interval.split("-"))
        excerpt = sources[int(document)][start:end].decode()
        print(f"\n{rank}. RFC {document}, байты [{start}, {end})")
        print(textwrap.fill(" ".join(excerpt.split()), width=95))
    print()